In [ ]:
import os
import json
import time
import requests
import pandas as pd
from dotenv import load_dotenv
from tqdm import tqdm
import psycopg2
from psycopg2.extras import Json

#### Authentification INPI

In [ ]:
# 1. Chargement des variables d'environnement
load_dotenv()

USERNAME = os.getenv("INPI_USERNAME")
PASSWORD = os.getenv("INPI_PASSWORD")

if not USERNAME or not PASSWORD:
    raise RuntimeError("❌ INPI_USERNAME ou INPI_PASSWORD manquant dans le .env")

# 2. Authentification et récupération du Token
LOGIN_URL = "https://registre-national-entreprises.inpi.fr/api/sso/login"

payload = {
    "username": USERNAME,
    "password": PASSWORD,
}

try:
    print("🔑 Tentative de connexion à l'INPI...")
    response = requests.post(LOGIN_URL, json=payload, timeout=15)
    response.raise_for_status()
    
    token = response.json().get("token")
    
    if not token:
        raise ValueError("Le format de réponse de l'API ne contient pas de token.")
        
    print(f"✅ Authentification réussie !")
    print(f"🎫 Token OK : {token[:20]}...")

    # 3. Configuration des Headers pour la suite
    HEADERS = {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json"
    }

except requests.exceptions.HTTPError as err:
    print(f"❌ Erreur HTTP lors de la connexion : {err}")
except Exception as e:
    print(f"❌ Erreur inattendue : {e}")

---

### Acquisition des bilans

#### Import des SIREN

In [ ]:
with open("siren_sample.json", "r", encoding="utf-8") as f:
    sirens_a_traiter = json.load(f)

print(f"📂 {len(sirens_a_traiter)} SIREN chargés depuis le fichier.")

#### Connection SQL

In [ ]:
load_dotenv()

DATABASE_URL = os.getenv("NEON_DATABASE_URL")

if not DATABASE_URL:
    raise ValueError("La variable NEON_DATABASE_URL n'est pas définie dans .env")

try:
    # On ajoute sslmode=require pour Neon
    conn = psycopg2.connect(DATABASE_URL)
    cur = conn.cursor()
    
    # Test pour confirmer la version de la BDD
    cur.execute("SELECT version();")
    db_version = cur.fetchone()
    print(f"✅ Connexion à Néon réussie !")
    print(f"🖥️ Version : {db_version[0][:30]}...")

except Exception as e:
    print(f"❌ Erreur de connexion : {e}")

#### Acquisition des bilans des 500 premiers SIREN

In [ ]:
def extraire_bilans_dictionnaire(data):
    """Transforme le JSON complexe de l'INPI en dictionnaire 'Key-Value'"""
    bilans = []
    if "bilansSaisis" in data:
        for b in data["bilansSaisis"]:
            liasses_dict = {}
            try:
                # On descend dans la structure spécifique de l'API RNE
                pages = b.get('bilanSaisi', {}).get('bilan', {}).get('detail', {}).get('pages', [])
                for page in pages:
                    for liasse in page.get('liasses', []):
                        code = liasse.get("code")
                        # On récupère m1 (valeur brute), par défaut 0
                        valeur = int(liasse.get("m1") or 0)
                        if code:
                            liasses_dict[code] = valeur
                
                bilans.append({
                    "siren": b.get("siren"),
                    "date_cloture": b.get("dateCloture"),
                    "confidentialite": b.get("confidentiality"),
                    "liasses": liasses_dict
                })
            except (KeyError, TypeError) as e:
                # Si un bilan est mal formé, on passe au suivant
                continue
    return bilans

# --- CONFIGURATION ---
BACKUP_DIR = "backup_bilans"
os.makedirs(BACKUP_DIR, exist_ok=True)

# # 1. SYNCHRONISATION : On vérifie ce qu'on a déjà (Local + Neon)

# local_files = {f.replace(".json", "") for f in os.listdir(BACKUP_DIR)}

# # Vérification BDD Neon
# cur.execute("SELECT DISTINCT siren FROM bilans_entreprises")
# neon_sirens = {row[0] for row in cur.fetchall()}

# # On ne traite que ce qui n'est NI en local, NI sur Neon
# deja_faits = local_files.union(neon_sirens)
# sirens_restants = [s for s in sirens_a_traiter if s not in deja_faits]

# print(f"📊 État des lieux :")
# print(f"📁 Archive locale : {len(local_files)} entreprises")
# print(f"☁️  Base Neon      : {len(neon_sirens)} entreprises")
# print(f"🚀 Reste à traiter : {len(sirens_restants)}")
# print("-" * 50)

# # 2. BOUCLE D'ACQUISITION
# for i, siren in enumerate(sirens_restants):
#     print(f"⏳ [{i+1}/{len(sirens_restants)}] Traitement : {siren}...", end="\r")
    
#     try:
#         url = f"https://registre-national-entreprises.inpi.fr/api/companies/{siren}/attachments"
#         res = requests.get(url, headers=HEADERS, timeout=20)
        
#         if res.status_code == 200:
#             data = res.json()
#             bilans_propres = extraire_bilans_dictionnaire(data)
            
#             # SAUVEGARDE LOCALE (Fichier individuel)
#             with open(f"{BACKUP_DIR}/{siren}.json", "w", encoding="utf-8") as f:
#                 json.dump(bilans_propres, f, indent=2, ensure_ascii=False)
            
#             # INJECTION NEON
#             if bilans_propres:
#                 insert_query = """
#                 INSERT INTO bilans_entreprises (siren, date_cloture, confidentialite, donnees_liasses)
#                 VALUES (%s, %s, %s, %s)
#                 ON CONFLICT (siren, date_cloture) DO NOTHING;
#                 """
#                 for b in bilans_propres:
#                     cur.execute(insert_query, (
#                         b['siren'], 
#                         b['date_cloture'], 
#                         b['confidentialite'], 
#                         Json(b['liasses'])
#                     ))
#                 conn.commit()
                
#         elif res.status_code == 429:
#             print(f"\n⚠️ Rate limit ! Pause de 60s...")
#             time.sleep(60)
#             # Optionnel: interrompre ici ou continuer
            
#     except Exception as e:
#         print(f"\n❌ Erreur sur {siren} : {e}")
#         conn.rollback()

#     # Respect du quota INPI
#     time.sleep(5.5)

# print("\n\n✅ Mission terminée. Ton dossier 'backup_bilans' et ta base Neon sont synchronisés.")

---

#### Augmentation des bilans à acquérir avec l'échantillon à 2000 SIREN

In [ ]:
# --- CONFIGURATION ---
BACKUP_DIR = "backup_bilans"
INPUT_FILE = "siren_batch_2000.json"

# 1. FILTRAGE : On ne télécharge que l'inconnu
with open(INPUT_FILE, 'r') as f:
    tous_sirens = json.load(f)

local_files = {f.replace(".json", "") for f in os.listdir(BACKUP_DIR)}
sirens_restants = [s for s in tous_sirens if s not in local_files]

print(f"📊 ANALYSE DU BATCH :")
print(f"✅ Déjà en local : {len(local_files)}")
print(f"🚀 À récupérer   : {len(sirens_restants)}")
print("-" * 50)

# 2. BOUCLE D'ACQUISITION
for i, siren in enumerate(sirens_restants):
    print(f"⏳ [{i+1}/{len(sirens_restants)}] SIREN: {siren}", end="\r")
    
    try:
        url = f"https://registre-national-entreprises.inpi.fr/api/companies/{siren}/attachments"
        res = requests.get(url, headers=HEADERS, timeout=20)
        
        if res.status_code == 200:
            data = res.json()
            bilans = extraire_bilans_dictionnaire(data)
            
            # Sauvegarde locale
            with open(f"{BACKUP_DIR}/{siren}.json", "w", encoding="utf-8") as f:
                json.dump(bilans, f, indent=2, ensure_ascii=False)
            
            # Injection Neon
            if bilans:
                for b in bilans:
                    cur.execute("""
                        INSERT INTO bilans_entreprises (siren, date_cloture, confidentialite, donnees_liasses)
                        VALUES (%s, %s, %s, %s)
                        ON CONFLICT (siren, date_cloture) DO NOTHING;
                    """, (b['siren'], b['date_cloture'], b['confidentialite'], Json(b['liasses'])))
                conn.commit()
        
        elif res.status_code == 429:
            print("\n⚠️ Pause forcée (Rate limit)...")
            time.sleep(60)
            
    except Exception as e:
        print(f"\n❌ Erreur {siren}: {e}")
        conn.rollback()

    time.sleep(5.5) # Tempo de sécurité pour l'INPI